# P0 → P3 · Unsupervised defect-type discovery (MVTec AD)

After the codebook flags suspicious patches, can we sort anomalous images by **kind** of defect with no labels?
Each image is described by its 10 most suspicious patches; images are clustered and compared with MVTec's
defect-type folders (NMI, ARI, accuracy; chance = shuffled labels).

| Descriptor | What it encodes |
|---|---|
| `topk_resid` | what the normal vocabulary *cannot* explain: patch minus its nearest word |
| `topk_feat` | what the suspicious patches look like |
| `cls` | whole-image embedding (baseline) |
| `mask_feat` | features inside the true defect mask (upper bound, uses labels) |

**Settings:** `oracle` clusters the truly anomalous images; `detected` clusters the images the detector flags,
so false alarms join as a "good" class. Reuses cached features and vocabularies from notebook 03:
about 10–15 min on a T4. Exploratory, not preregistered.

## 1 · Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: DINOv2 extraction will be slow"

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Get the code. Option A: GitHub (default). Option B: set REPO_URL = "" to install
# from bovw-forensics.zip uploaded to MyDrive/bovw-forensics/.
REPO_URL = "https://github.com/M3R2T2L2/bovw_forensics.git"
ZIP_PATH = "/content/drive/MyDrive/bovw-forensics/bovw-forensics.zip"

import os, shutil, subprocess, zipfile
shutil.rmtree("/content/bovw_forensics", ignore_errors=True)
if REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/bovw_forensics"], check=True)
else:
    zipfile.ZipFile(ZIP_PATH).extractall("/content/")
    os.rename("/content/bovw-forensics", "/content/bovw_forensics")

%cd /content/bovw_forensics
!pip install -q -e ".[dev]"   # torch, torchvision, transformers come preinstalled on Colab

In [ ]:
# 30-second check that the torch-free core works in this runtime
!python -m pytest -q

In [ ]:
# STL-10 on Drive: restore it here (skips the ~45 min download), or save it after the first download.
import os, shutil
DRIVE_STL = "/content/drive/MyDrive/bovw-forensics/data/stl10_binary"
LOCAL_STL = "/content/data/stl10_binary"
if os.path.isdir(DRIVE_STL) and not os.path.isdir(LOCAL_STL):
    shutil.copytree(DRIVE_STL, LOCAL_STL); print("Restored STL-10 from Drive")
elif os.path.isdir(LOCAL_STL) and not os.path.isdir(DRIVE_STL):
    shutil.copytree(LOCAL_STL, DRIVE_STL); print("Saved STL-10 to Drive for future runtimes")
else:
    print("STL-10 will download on first use; re-run this cell afterwards to save it to Drive"
          if not os.path.isdir(LOCAL_STL) else "STL-10 already local and on Drive")

## 2 · Data
The archive is kept on Drive so it downloads only once; each new runtime extracts it locally (~3–5 min).

In [ ]:
import os, subprocess
from bovw import ad_data
DRIVE_TAR = "/content/drive/MyDrive/bovw-forensics/data/mvtec_anomaly_detection.tar.xz"
LOCAL_ROOT = "/content/mvtec"

# Set to True only after accepting the licence on the MVTec page above.
I_ACCEPT_MVTEC_LICENCE = False

if not os.path.isfile(DRIVE_TAR) or os.path.getsize(DRIVE_TAR) < 1e9:   # missing, or a stub from a failed download
    if not I_ACCEPT_MVTEC_LICENCE:
        raise SystemExit("Accept the MVTec AD licence, then set I_ACCEPT_MVTEC_LICENCE = True "
                         "(or upload mvtec_anomaly_detection.tar.xz to MyDrive/bovw-forensics/data/).")
# Downloads only if missing or corrupt; verifies the SHA-256 (hashing ~5 GB takes about a minute).
ad_data.ensure_archive(DRIVE_TAR)

if not os.path.isdir(f"{LOCAL_ROOT}/bottle"):
    os.makedirs(LOCAL_ROOT, exist_ok=True)
    subprocess.run(["tar", "-xf", DRIVE_TAR, "-C", LOCAL_ROOT], check=True)
print(sorted(os.listdir(LOCAL_ROOT)))

## 3 · Run

In [ ]:
from bovw.sweep import load_config
from bovw import discovery

cfg = load_config("configs/p0_defect_discovery_mvtec.yaml")
cfg["data"]["root"] = LOCAL_ROOT
df = discovery.run(cfg)

## 4 · Results
`nmi_above_chance` is the headline: NMI minus the shuffled-label NMI of the same clustering.

In [ ]:
s = discovery.summary(df)
s.to_csv(f"{cfg['results_dir']}/{cfg['name']}_summary.csv")
s.round(3)

In [ ]:
# Per category, oracle setting, k-means
o = df[(df.setting == "oracle") & (df.algorithm == "kmeans")]
t = o.pivot_table(index=["category"], columns="descriptor", values="nmi_above_chance").round(3)
t.insert(0, "n_types", o.groupby("category").n_types.first())
t.to_csv(f"{cfg['results_dir']}/{cfg['name']}_per_category.csv")
t

## 5 · Look at the clusters
Rows = discovered clusters; each tile is titled with its true defect type.

In [ ]:
import matplotlib.pyplot as plt
from bovw import ad_data

CATEGORY = "bottle"   # try metal_nut, hazelnut, pill, cable
idx, pred, types = discovery.cluster_category(cfg, CATEGORY, "topk_resid")
split = ad_data.load_mvtec(LOCAL_ROOT, CATEGORY, 224, 64)
n_cl, per = pred.max() + 1, 6
fig, axes = plt.subplots(n_cl, per, figsize=(1.9 * per, 2.1 * n_cl), squeeze=False)
for c in range(n_cl):
    members = idx[pred == c][:per]
    for j in range(per):
        ax = axes[c][j]; ax.set_axis_off()
        if j < len(members):
            ax.imshow(split.test_images[members[j]])
            ax.set_title(types[pred == c][j], fontsize=7)
    axes[c][0].text(-0.1, 0.5, f"cluster {c}\n(n={int((pred == c).sum())})", transform=axes[c][0].transAxes,
                    ha="right", va="center", fontsize=8)
fig.tight_layout()
fig.savefig(f"{cfg['results_dir']}/{cfg['name']}_{CATEGORY}_clusters.png", dpi=110, bbox_inches="tight")